# AI Mathematical Olympiad — Full Pipeline

**MCTS-based Math Solver with Symbolic Verification**

This notebook runs the complete pipeline:
1. Install dependencies & verify GPU
2. Data pipeline — stream OpenMathReasoning TIR → parquet
3. SFT training — QLoRA fine-tune NuminaMath-7B-TIR
4. MCTS-RL training — generate verified solutions via MCTS, fine-tune on them
5. Evaluation — compare base vs SFT vs MCTS-RL
6. Publication-quality plots
7. Download trained adapters

**Before running:** Runtime → Change runtime type → **T4 GPU**

---
## 1. Setup & GPU Verification

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl \
    matplotlib sympy pandas pyarrow tqdm

In [ ]:
import torch
import os

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

GPU_NAME  = torch.cuda.get_device_name(0)
VRAM_GB   = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")
print(f"PyTorch: {torch.__version__}")

# Directories
os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/models/sft", exist_ok=True)
os.makedirs("/content/models/mcts_rl", exist_ok=True)
os.makedirs("/content/experiments", exist_ok=True)

---
## 2. Data Pipeline — Stream OpenMathReasoning TIR

Streams from `nvidia/OpenMathReasoning` (TIR split), transforms to chat format,
and writes to a parquet file. Only the current batch is in memory.

In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from tqdm.auto import tqdm

DATA_LIMIT = 1_000_000  # Set to None for full 1.7M rows
BATCH_SIZE = 5_000
OUTPUT_PATH = "/content/data/processed/math_reasoning_tir.parquet"

SCHEMA = pa.schema([
    ("problem", pa.string()),
    ("solution", pa.string()),
    ("expected_answer", pa.string()),
    ("difficulty", pa.string()),
    ("messages", pa.list_(pa.struct([
        ("role", pa.string()),
        ("content", pa.string()),
    ]))),
])


def build_tir_messages(problem, solution):
    return [
        {"role": "system", "content": ""},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": solution},
    ]


def format_tir_chat(problem, solution):
    return (
        f"<|system|>\n<|end|>\n"
        f"<|user|>\n{problem}<|end|>\n"
        f"<|assistant|>\n{solution}<|end|>"
    )


def transform_row(example):
    problem = example["problem"]
    solution = example["generated_solution"]
    return {
        "problem": problem,
        "solution": solution,
        "expected_answer": example["expected_answer"],
        "difficulty": example.get("problem_source", "unknown"),
        "messages": build_tir_messages(problem, solution),
    }


print("Streaming nvidia/OpenMathReasoning (tir split)...")
ds = load_dataset("nvidia/OpenMathReasoning", split="tir", streaming=True)

writer = None
batch = []
rows_written = 0

for example in tqdm(ds, desc="Processing", total=DATA_LIMIT):
    if DATA_LIMIT and rows_written + len(batch) >= DATA_LIMIT:
        break
    batch.append(transform_row(example))

    if len(batch) >= BATCH_SIZE:
        table = pa.Table.from_pylist(batch, schema=SCHEMA)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
        writer.write_table(table)
        rows_written += len(batch)
        batch.clear()
        del table
        gc.collect()
        print(f"  Written {rows_written:,} rows")

if batch:
    table = pa.Table.from_pylist(batch, schema=SCHEMA)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, SCHEMA, compression='snappy')
    writer.write_table(table)
    rows_written += len(batch)
    batch.clear()
    del table

if writer:
    writer.close()

del ds, writer
gc.collect()

final_count = pq.read_metadata(OUTPUT_PATH).num_rows
print(f"\nDone: {final_count:,} rows -> {OUTPUT_PATH}")

In [ ]:
# Quick peek at the data
import pandas as pd
df_peek = pd.read_parquet(OUTPUT_PATH, columns=["problem", "expected_answer", "difficulty"])
print(f"Total rows: {len(df_peek):,}")
print(f"\nDifficulty distribution:")
print(df_peek["difficulty"].value_counts().head(10))
print(f"\nSample problem:\n{df_peek.iloc[0]['problem'][:300]}...")

---
## 3. SFT Training — QLoRA Fine-Tuning

Fine-tunes NuminaMath-7B-TIR on the OpenMathReasoning TIR data using:
- 4-bit NF4 quantization + double quant
- LoRA (r=16, alpha=32) on attention projections
- Gradient checkpointing
- SFTTrainer with sequence packing
- Paged AdamW 8-bit optimizer

In [ ]:
from datasets import Dataset
import pyarrow.parquet as pq

# Prepare text dataset for SFT — streamed from parquet to avoid memory blowup
SFT_LIMIT = 1_000_000  # Rows to use for SFT

def sft_generator():
    pf = pq.ParquetFile(OUTPUT_PATH)
    count = 0
    for batch in pf.iter_batches(batch_size=10_000, columns=["problem", "solution"]):
        for i in range(len(batch)):
            if SFT_LIMIT and count >= SFT_LIMIT:
                return
            yield {"text": format_tir_chat(
                batch.column("problem")[i].as_py(),
                batch.column("solution")[i].as_py(),
            )}
            count += 1

sft_dataset = Dataset.from_generator(sft_generator)
gc.collect()

sft_dataset = sft_dataset.train_test_split(test_size=0.1, seed=42)
print(f"SFT Train: {len(sft_dataset['train']):,} | Eval: {len(sft_dataset['test']):,}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "AI-MO/NuminaMath-7B-TIR"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map={"":0},
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
))

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.enable_input_require_grads()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded: {MODEL_ID}")
print(f"Trainable: {trainable_params:,} / {total_params:,} ({trainable_params/total_params:.2%})")
print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB / {VRAM_GB:.1f} GB")

In [ ]:
from trl import SFTTrainer, SFTConfig

SFT_OUTPUT = "/content/models/sft"
SFT_EPOCHS = 3

sft_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["test"],
    args=SFTConfig(
        output_dir=f"{SFT_OUTPUT}/checkpoints",
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        bf16=True,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=500,
        save_steps=500,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        max_length=2048,
        dataset_text_field="text",
        packing=True,
    ),
)

print("Starting SFT training...")
sft_result = sft_trainer.train()
print(f"SFT training complete. Final loss: {sft_result.training_loss:.4f}")

In [ ]:
import math
import shutil

# Evaluate
sft_eval = sft_trainer.evaluate()
sft_eval_loss = sft_eval["eval_loss"]

# Extract loss histories
sft_log = sft_trainer.state.log_history
sft_train_losses = [(l["step"], l["loss"]) for l in sft_log if "loss" in l]
sft_eval_losses = [(l["step"], l["eval_loss"]) for l in sft_log if "eval_loss" in l]

print(f"SFT Eval Loss:  {sft_eval_loss:.4f}")
print(f"SFT Perplexity: {math.exp(sft_eval_loss):.2f}")

# Save SFT adapter
SFT_ADAPTER = f"{SFT_OUTPUT}/lora_adapter"
model.save_pretrained(SFT_ADAPTER)
tokenizer.save_pretrained(SFT_ADAPTER)
print(f"SFT adapter saved to {SFT_ADAPTER}")

# Delete checkpoints to free disk space
ckpt_dir = f"{SFT_OUTPUT}/checkpoints"
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"Deleted checkpoints at {ckpt_dir} to free disk space")

---
## 4. MCTS Search Engine & Verification

Core components:
- **MCTS** with UCT selection, LLM expansion, symbolic pruning
- **Verifiers**: symbolic (SymPy), code execution (subprocess), answer back-substitution
- **Weighted majority voting** for final answer selection

In [ ]:
import re
import math
import time
import subprocess
import sys
import sympy
from typing import List, Optional, Dict, Tuple, Any
from dataclasses import dataclass
from abc import ABC, abstractmethod


# ===== Verifiers =====

class BaseVerifier(ABC):
    @abstractmethod
    def verify(self, text: str) -> Tuple[bool, str]:
        pass


class NoOpVerifier(BaseVerifier):
    def verify(self, text: str) -> Tuple[bool, str]:
        return True, "no_verify"


class SymbolicVerifier(BaseVerifier):
    """Checks mathematical consistency using SymPy.
    Detects equation contradictions and cross-step variable inconsistencies."""

    def verify(self, text: str) -> Tuple[bool, str]:
        equations = re.findall(r'\$(.*?)\$', text)
        boxed = re.findall(r'\\boxed\{(.+?)\}', text)

        for err in self._check_equations(equations):
            return False, err
        for b in boxed:
            if "=" in b:
                for err in self._check_equations([b]):
                    return False, f"Boxed contradiction: {err}"

        assignments = self._extract_assignments(equations + boxed)
        for err in self._check_variable_consistency(assignments):
            return False, err
        return True, "passed"

    def _check_equations(self, equations):
        errors = []
        for eq in equations:
            try:
                if "=" in eq and "==" not in eq and "\\neq" not in eq and "!=" not in eq:
                    parts = eq.split("=", 1)
                    if len(parts) != 2: continue
                    lhs, rhs = parts[0].strip(), parts[1].strip()
                    if not lhs or not rhs: continue
                    diff = sympy.simplify(f"({lhs}) - ({rhs})")
                    if diff != 0 and diff.is_number:
                        errors.append(f"Contradiction: {lhs} != {rhs}")
            except Exception:
                continue
        return errors

    def _extract_assignments(self, equations):
        assignments = {}
        var_pat = re.compile(r'^([a-zA-Z_]\w*)\s*$')
        for eq in equations:
            try:
                if "=" not in eq or "==" in eq or "\\neq" in eq: continue
                parts = eq.split("=", 1)
                if len(parts) != 2: continue
                lhs, rhs = parts[0].strip(), parts[1].strip()
                if not lhs or not rhs: continue
                for var_side, val_side in [(lhs, rhs), (rhs, lhs)]:
                    if var_pat.match(var_side):
                        try:
                            value = sympy.sympify(val_side)
                            if value.is_number:
                                assignments.setdefault(var_side, []).append(value)
                        except Exception:
                            pass
            except Exception:
                continue
        return assignments

    def _check_variable_consistency(self, assignments):
        errors = []
        for var, values in assignments.items():
            if len(values) < 2: continue
            first = values[0]
            for v in values[1:]:
                diff = sympy.simplify(first - v)
                if diff != 0 and diff.is_number:
                    errors.append(f"Variable '{var}' assigned conflicting values: {first} and {v}")
                    break
        return errors


class CodeExecutionVerifier(BaseVerifier):
    """Executes Python code blocks with persistent state and timeout."""

    def __init__(self, timeout=10):
        self.timeout = timeout

    def verify(self, text: str) -> Tuple[bool, str]:
        blocks = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
        if not blocks:
            return True, "no_code"
        combined = "\n".join(blocks)
        try:
            result = subprocess.run(
                [sys.executable, "-c", combined],
                capture_output=True, text=True, timeout=self.timeout,
            )
            if result.returncode != 0:
                return False, f"Code error: {result.stderr.strip().split(chr(10))[-1]}"
            return True, "code_passed"
        except subprocess.TimeoutExpired:
            return False, "timeout"
        except Exception as e:
            return False, str(e)


class AnswerVerifier(BaseVerifier):
    """Back-substitutes boxed answer into problem equations."""

    def verify(self, text: str) -> Tuple[bool, str]:
        answer_match = re.search(r'\\boxed\{(.+?)\}', text)
        if not answer_match:
            return True, "no_answer_yet"
        try:
            answer_val = sympy.sympify(answer_match.group(1))
        except Exception:
            return True, "answer_not_parseable"

        lines = text.split("\n")
        problem_text = lines[0] if lines else text
        equations = re.findall(r'\$(.*?)\$', problem_text)
        if not equations:
            return True, "no_equations_in_problem"

        for eq in equations:
            try:
                if "=" not in eq or "==" in eq or "\\neq" in eq: continue
                parts = eq.split("=", 1)
                if len(parts) != 2: continue
                lhs_str, rhs_str = parts[0].strip(), parts[1].strip()
                if not lhs_str or not rhs_str: continue
                lhs = sympy.sympify(lhs_str)
                rhs = sympy.sympify(rhs_str)
                free_vars = lhs.free_symbols | rhs.free_symbols
                if len(free_vars) != 1: continue
                var = free_vars.pop()
                residual = sympy.simplify((lhs - rhs).subs(var, answer_val))
                if residual != 0 and residual.is_number:
                    return False, f"Back-substitution failed for {var}={answer_val}"
            except Exception:
                continue
        return True, "back_substitution_passed"


class CompositeVerifier(BaseVerifier):
    def __init__(self, verifiers):
        self.verifiers = verifiers

    def verify(self, text: str) -> Tuple[bool, str]:
        for v in self.verifiers:
            ok, reason = v.verify(text)
            if not ok:
                return False, f"{v.__class__.__name__}: {reason}"
        return True, "all_passed"


def create_verifier(mode="both"):
    if mode == "none":     return NoOpVerifier()
    elif mode == "symbolic": return SymbolicVerifier()
    elif mode == "code":    return CodeExecutionVerifier()
    elif mode == "both":
        return CompositeVerifier([SymbolicVerifier(), CodeExecutionVerifier(), AnswerVerifier()])
    raise ValueError(f"Unknown mode: {mode}")


print("Verifiers loaded: NoOp, Symbolic, CodeExecution, Answer, Composite")

In [ ]:
# ===== LLM Generator =====

class TransformersGenerator:
    """Generate solution candidates using the model on GPU."""

    def __init__(self, model_id="AI-MO/NuminaMath-7B-TIR", adapter_path=None):
        from peft import PeftModel

        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            ),
            device_map={"":0}, trust_remote_code=True,
        )

        if adapter_path:
            self.model = PeftModel.from_pretrained(self.model, adapter_path)

        self.model.eval()
        self.model_id = model_id
        self.device = "cuda"

    def get_candidates(self, context, n=3):
        inputs = self.tokenizer(context, return_tensors="pt").to(self.device)
        candidates = []
        for _ in range(n):
            with torch.no_grad():
                out = self.model.generate(
                    **inputs, max_new_tokens=512, temperature=0.7,
                    do_sample=True, pad_token_id=self.tokenizer.eos_token_id,
                )
            text = self.tokenizer.decode(
                out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
            candidates.append(text)
            # MEMORY FIX: Delete output tensor immediately
            del out
        # MEMORY FIX: Delete inputs and clear cache after batch
        del inputs
        torch.cuda.empty_cache()
        return candidates


print("Generator loaded.")

In [ ]:
# ===== MCTS Engine =====

@dataclass
class MCTSMetrics:
    total_nodes: int = 0
    pruned_nodes: int = 0
    total_tokens: int = 0
    search_time: float = 0.0
    iterations_completed: int = 0
    answer_candidates: int = 0
    unique_answers: int = 0

    @property
    def prune_rate(self):
        return self.pruned_nodes / self.total_nodes if self.total_nodes else 0.0

    def to_dict(self):
        return {
            "total_nodes": self.total_nodes, "pruned_nodes": self.pruned_nodes,
            "prune_rate": round(self.prune_rate, 4), "total_tokens": self.total_tokens,
            "search_time": round(self.search_time, 2),
            "iterations": self.iterations_completed,
            "answer_candidates": self.answer_candidates,
            "unique_answers": self.unique_answers,
        }


class MCTSNode:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = []
        self.visits = 0
        self.value = 0.0

    def uct(self, c=1.41):
        if self.visits == 0: return float('inf')
        return (self.value / self.visits) + c * math.sqrt(math.log(self.parent.visits) / self.visits)

    @property
    def depth(self):
        d, node = 0, self
        while node.parent:
            d += 1; node = node.parent
        return d


class AIMO_MCTS:
    """MCTS with symbolic verification and weighted majority voting."""

    def __init__(self, generator, verifier=None, uct_constant=1.41):
        self.generator = generator
        self.verifier = verifier or create_verifier("both")
        self.uct_constant = uct_constant
        self.metrics = MCTSMetrics()
        self._tokenizer = None

    def _count_tokens(self, text):
        if self._tokenizer is None:
            tok = getattr(self.generator, 'tokenizer', None)
            self._tokenizer = tok if tok and hasattr(tok, 'encode') else False
        if self._tokenizer:
            try: return len(self._tokenizer.encode(text))
            except: pass
        return len(text.split())

    def search(self, problem, iterations=10, max_depth=10, candidates_per_node=3):
        self.metrics = MCTSMetrics()
        start = time.time()
        root = MCTSNode(state=problem)

        for i in range(iterations):
            node = root
            while node.children:
                node = max(node.children, key=lambda c: c.uct(self.uct_constant))

            if node.depth < max_depth:
                candidates = self.generator.get_candidates(node.state, n=candidates_per_node)
                for candidate in candidates:
                    self.metrics.total_nodes += 1
                    self.metrics.total_tokens += self._count_tokens(candidate)
                    full_state = node.state + "\n" + candidate
                    is_valid, _ = self.verifier.verify(full_state)
                    if is_valid:
                        node.children.append(MCTSNode(state=full_state, parent=node))
                    else:
                        self.metrics.pruned_nodes += 1
                # MEMORY FIX: Delete candidates after processing
                del candidates

            reward = self._score(node.state)
            current = node
            while current:
                current.visits += 1
                current.value += reward
                current = current.parent
            self.metrics.iterations_completed = i + 1
            
            # MEMORY FIX: Periodic garbage collection during long searches
            if (i + 1) % 5 == 0:
                gc.collect()

        self.metrics.search_time = time.time() - start
        final_answer = self._select_answer(root)
        
        # MEMORY FIX: Clear the tree after extracting answer
        self._clear_tree(root)
        del root
        gc.collect()
        
        return final_answer

    def _clear_tree(self, node):
        """Recursively clear tree nodes to free memory."""
        for child in node.children:
            self._clear_tree(child)
        node.children.clear()
        node.state = None
        node.parent = None

    def _collect_answers(self, node):
        results = []
        if not node.children:
            answer = self._extract_boxed(node.state)
            if answer:
                results.append({"answer": answer, "state": node.state, "visits": node.visits})
            return results
        for child in node.children:
            results.extend(self._collect_answers(child))
        return results

    def _select_answer(self, root):
        candidates = self._collect_answers(root)
        if not candidates:
            node = root
            while node.children:
                node = max(node.children, key=lambda c: c.visits)
            return node.state

        self.metrics.answer_candidates = len(candidates)
        answer_weights = {}
        best_state = {}
        for c in candidates:
            norm = self._normalize_answer(c["answer"])
            answer_weights[norm] = answer_weights.get(norm, 0.0) + c["visits"]
            prev = best_state.get(norm)
            if prev is None or c["visits"] > prev[0]:
                best_state[norm] = (c["visits"], c["state"])

        self.metrics.unique_answers = len(answer_weights)
        winner = max(answer_weights, key=answer_weights.get)
        return best_state[winner][1]

    def _extract_boxed(self, text):
        match = re.search(r'\\boxed\{(.+?)\}', text)
        return match.group(1) if match else None

    def _normalize_answer(self, answer):
        answer = re.sub(r'\s+', '', answer.strip())
        try:
            num = float(answer)
            return str(int(num)) if num == int(num) else f"{num:.6f}".rstrip('0').rstrip('.')
        except ValueError:
            return answer.lower()

    def _score(self, state):
        if self._has_answer(state): return 1.0
        if "```python" in state and "```output" in state: return 0.5
        if "```python" in state: return 0.3
        return 0.1

    def _has_answer(self, text):
        return bool(re.search(r'\\boxed\{.+?\}', text)) or "final answer" in text.lower()

    def get_metrics(self):
        return self.metrics.to_dict()


print("MCTS engine loaded.")

### Quick MCTS test on a sample problem

In [ ]:
# Free the SFT trainer model to make room for the generator
del sft_trainer, model
gc.collect()
torch.cuda.empty_cache()

# Load generator with SFT adapter
gen = TransformersGenerator(model_id=MODEL_ID, adapter_path=SFT_ADAPTER)
mcts = AIMO_MCTS(generator=gen, verifier=create_verifier("both"))

# Test on first problem from dataset
test_df = pd.read_parquet(OUTPUT_PATH, columns=["problem", "expected_answer"]).head(1)
test_problem = test_df.iloc[0]["problem"]
test_expected = test_df.iloc[0]["expected_answer"]

print(f"Problem: {test_problem[:200]}...")
print(f"Expected: {test_expected}")
print("\nRunning MCTS search (3 iterations)...")

solution = mcts.search(test_problem, iterations=3, candidates_per_node=2)
metrics = mcts.get_metrics()

print(f"\nMetrics: {metrics}")
print(f"\nSolution (last 300 chars):\n{solution[-300:]}")

---
## 5. MCTS-RL Training

Generate verified solutions using MCTS, then fine-tune on only the correct ones.
This creates a second LoRA adapter on top of (or replacing) the SFT adapter.

In [ ]:
import json

# ===== Metrics functions =====

def extract_answer(text):
    patterns = [
        r'\\boxed\{([^}]+)\}',
        r'final answer[:\s]*\$?([^\n\$]+)\$?',
        r'answer is[:\s]*\$?([^\n\$]+)\$?',
        r'= ([0-9]+)\s*$',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            answer = match.group(1).strip()
            return re.sub(r'[\\${}]', '', answer).strip()
    return None


def normalize_answer(answer):
    if not answer: return ""
    answer = re.sub(r'\s+', '', answer.lower().strip())
    answer = re.sub(r'[\\${}]', '', answer)
    try:
        num = float(answer)
        return str(int(num)) if num == int(num) else f"{num:.6f}".rstrip('0').rstrip('.')
    except ValueError:
        return answer


def evaluate_answer(predicted, expected):
    pred_answer = extract_answer(predicted)
    exp_answer = normalize_answer(expected)
    pred_normalized = normalize_answer(pred_answer) if pred_answer else ""
    is_correct = pred_normalized == exp_answer
    if not is_correct and pred_normalized and exp_answer:
        try:
            is_correct = abs(float(pred_normalized) - float(exp_answer)) < 1e-6
        except ValueError:
            pass
    return {
        "correct": is_correct,
        "predicted_raw": pred_answer,
        "predicted_normalized": pred_normalized,
        "expected_normalized": exp_answer,
    }


def compute_aggregate_metrics(results):
    if not results: return {}
    n_correct = sum(1 for r in results if r.get("correct", False))
    n_total = len(results)
    total_tokens = sum(r.get("metrics", {}).get("total_tokens", 0) for r in results)
    total_time = sum(
        r.get("metrics", {}).get("solve_time", 0) or r.get("metrics", {}).get("search_time", 0)
        for r in results
    )
    pruned = sum(r.get("metrics", {}).get("pruned_nodes", 0) for r in results)
    total_nodes = sum(r.get("metrics", {}).get("total_nodes", 0) for r in results)
    return {
        "accuracy": round(n_correct / n_total, 4) if n_total else 0,
        "n_correct": n_correct, "n_total": n_total,
        "avg_tokens": round(total_tokens / n_total, 2) if n_total else 0,
        "avg_time": round(total_time / n_total, 2) if n_total else 0,
        "total_tokens": total_tokens, "total_time": round(total_time, 2),
        "prune_rate": round(pruned / total_nodes, 4) if total_nodes else 0,
    }


print("Evaluation metrics loaded.")

In [ ]:
# Generate verified solutions using MCTS
MCTS_RL_LIMIT = 200      # Problems to attempt
N_CANDIDATES = 3          # MCTS runs per problem
MCTS_ITERATIONS = 5       # Iterations per MCTS run
MCTS_RL_OUTPUT = "/content/models/mcts_rl"

problems_df = pd.read_parquet(OUTPUT_PATH).head(MCTS_RL_LIMIT)

verified_solutions = []
stats = {"total": 0, "correct": 0}

for idx, row in problems_df.iterrows():
    problem = row["problem"]
    expected = str(row["expected_answer"])
    stats["total"] += 1

    correct_solutions = []
    for _ in range(N_CANDIDATES):
        solution = mcts.search(problem, iterations=MCTS_ITERATIONS, candidates_per_node=2)
        result = evaluate_answer(solution, expected)
        m = mcts.get_metrics()
        if result["correct"]:
            correct_solutions.append({
                "solution": solution,
                "prune_rate": m.get("prune_rate", 0),
            })
        # MEMORY FIX: Delete solution text after evaluation
        del solution, result, m

    if correct_solutions:
        stats["correct"] += 1
        best = max(correct_solutions, key=lambda s: s["prune_rate"])
        verified_solutions.append({
            "text": format_tir_chat(problem, best["solution"]),
            "expected_answer": expected,
        })
    
    # MEMORY FIX: Clear solutions list and run gc every iteration
    del correct_solutions
    gc.collect()
    
    # MEMORY FIX: More aggressive cleanup every 10 problems
    if (idx + 1) % 10 == 0:
        torch.cuda.empty_cache()

    if (idx + 1) % 20 == 0:
        print(f"  [{idx+1}/{MCTS_RL_LIMIT}] Verified: {stats['correct']}/{stats['total']}")

print(f"\nTotal verified: {stats['correct']}/{stats['total']} ({stats['correct']/max(stats['total'],1):.1%})")

# Save verified solutions
os.makedirs(MCTS_RL_OUTPUT, exist_ok=True)
with open(f"{MCTS_RL_OUTPUT}/verified_solutions.json", "w") as f:
    json.dump(verified_solutions, f, indent=2)
print(f"Saved {len(verified_solutions)} verified solutions")

In [ ]:
# Fine-tune on verified solutions (MCTS-RL stage)
if len(verified_solutions) >= 3:
    # Free generator model
    del gen, mcts
    gc.collect()
    torch.cuda.empty_cache()

    # Load base model + merge SFT adapter
    from peft import PeftModel

    rl_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map={"":0}, trust_remote_code=True,
    )
    rl_model = prepare_model_for_kbit_training(rl_model)

    # Merge SFT adapter
    rl_model = PeftModel.from_pretrained(rl_model, SFT_ADAPTER)
    rl_model = rl_model.merge_and_unload()
    rl_model = prepare_model_for_kbit_training(rl_model)

    # New LoRA for RL stage
    rl_model = get_peft_model(rl_model, LoraConfig(
        r=8, lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    ))
    rl_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    rl_model.enable_input_require_grads()

    # Prepare verified dataset
    rl_dataset = Dataset.from_list(verified_solutions)
    rl_split = rl_dataset.train_test_split(test_size=0.1, seed=42)
    print(f"MCTS-RL Train: {len(rl_split['train'])} | Eval: {len(rl_split['test'])}")

    rl_trainer = SFTTrainer(
        model=rl_model,
        processing_class=tokenizer,
        train_dataset=rl_split["train"],
        eval_dataset=rl_split["test"],
        args=SFTConfig(
            output_dir=f"{MCTS_RL_OUTPUT}/checkpoints",
            num_train_epochs=2,
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=1e-4,
            warmup_ratio=0.1,
            weight_decay=0.01,
            bf16=True,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=50,
            save_steps=50,
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            report_to="none",
            optim="paged_adamw_8bit",
            max_length=2048,
            dataset_text_field="text",
            packing=True,
        ),
    )

    print("Starting MCTS-RL training...")
    rl_result = rl_trainer.train()

    rl_eval = rl_trainer.evaluate()
    rl_eval_loss = rl_eval["eval_loss"]
    rl_log = rl_trainer.state.log_history
    rl_train_losses = [(l["step"], l["loss"]) for l in rl_log if "loss" in l]
    rl_eval_losses = [(l["step"], l["eval_loss"]) for l in rl_log if "eval_loss" in l]

    print(f"MCTS-RL Eval Loss:  {rl_eval_loss:.4f}")
    print(f"MCTS-RL Perplexity: {math.exp(rl_eval_loss):.2f}")

    MCTS_RL_ADAPTER = f"{MCTS_RL_OUTPUT}/lora_adapter"
    rl_model.save_pretrained(MCTS_RL_ADAPTER)
    tokenizer.save_pretrained(MCTS_RL_ADAPTER)
    print(f"MCTS-RL adapter saved to {MCTS_RL_ADAPTER}")

    # Delete checkpoints to free disk space
    rl_ckpt_dir = f"{MCTS_RL_OUTPUT}/checkpoints"
    if os.path.exists(rl_ckpt_dir):
        shutil.rmtree(rl_ckpt_dir)
        print(f"Deleted checkpoints at {rl_ckpt_dir} to free disk space")

    del rl_trainer, rl_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"Only {len(verified_solutions)} verified solutions — skipping MCTS-RL training.")
    rl_train_losses, rl_eval_losses = [], []
    MCTS_RL_ADAPTER = None

---
## 6. Evaluation — Compare Base vs SFT vs MCTS-RL

Runs all three models on a held-out set of problems and compares accuracy,
compute cost (tokens), search time, and prune rates.

In [ ]:
EVAL_LIMIT = 20  # Number of problems for evaluation

eval_df = pd.read_parquet(OUTPUT_PATH).tail(EVAL_LIMIT)  # Use last N rows (unseen)
eval_df = eval_df.reset_index(drop=True)
print(f"Evaluating on {len(eval_df)} problems")

def run_evaluation(solver, method_name, problems_df):
    """Run a solver on all problems and collect results."""
    results = []
    for idx, row in problems_df.iterrows():
        problem = row["problem"]
        expected = str(row["expected_answer"])
        try:
            solution = solver.search(problem, iterations=5, candidates_per_node=2)
            metrics = solver.get_metrics()
            eval_result = evaluate_answer(solution, expected)
            result = {
                "method": method_name, "problem_idx": int(idx),
                "expected": expected, "predicted": eval_result["predicted_raw"],
                "correct": eval_result["correct"], "metrics": metrics,
            }
            mark = "CORRECT" if eval_result["correct"] else "WRONG"
            print(f"  [{method_name}] Problem {idx+1}: {mark} (expected={expected}, got={eval_result['predicted_raw']})")
            # MEMORY FIX: Delete intermediate variables
            del solution, metrics, eval_result
        except Exception as e:
            result = {"method": method_name, "problem_idx": int(idx), "error": str(e), "correct": False}
            print(f"  [{method_name}] Problem {idx+1}: ERROR ({e})")
        results.append(result)
        
        # MEMORY FIX: Periodic cleanup during evaluation
        if (idx + 1) % 5 == 0:
            gc.collect()
            torch.cuda.empty_cache()
    
    return results

In [ ]:
all_results = {}

# --- Base model (no fine-tuning) ---
print("\n" + "="*60 + "\nEvaluating: Base Model\n" + "="*60)
base_gen = TransformersGenerator(model_id=MODEL_ID)
base_mcts = AIMO_MCTS(generator=base_gen, verifier=create_verifier("symbolic"))
base_results = run_evaluation(base_mcts, "NuminaMath (base)", eval_df)
all_results["NuminaMath (base)"] = {
    "results": base_results,
    "aggregate": compute_aggregate_metrics(base_results),
}
# MEMORY FIX: More thorough cleanup
del base_gen, base_mcts, base_results
gc.collect()
torch.cuda.empty_cache()
# MEMORY FIX: Give GPU time to release memory
time.sleep(2)

# --- SFT model ---
print("\n" + "="*60 + "\nEvaluating: SFT Model\n" + "="*60)
sft_gen = TransformersGenerator(model_id=MODEL_ID, adapter_path=SFT_ADAPTER)
sft_mcts = AIMO_MCTS(generator=sft_gen, verifier=create_verifier("symbolic"))
sft_results = run_evaluation(sft_mcts, "SFT", eval_df)
all_results["SFT"] = {
    "results": sft_results,
    "aggregate": compute_aggregate_metrics(sft_results),
}
del sft_gen, sft_mcts, sft_results
gc.collect()
torch.cuda.empty_cache()
time.sleep(2)

# --- MCTS-RL model ---
if MCTS_RL_ADAPTER:
    print("\n" + "="*60 + "\nEvaluating: MCTS-RL Model\n" + "="*60)
    rl_gen = TransformersGenerator(model_id=MODEL_ID, adapter_path=MCTS_RL_ADAPTER)
    rl_mcts = AIMO_MCTS(generator=rl_gen, verifier=create_verifier("both"))
    rl_results = run_evaluation(rl_mcts, "MCTS-RL (ours)", eval_df)
    all_results["MCTS-RL (ours)"] = {
        "results": rl_results,
        "aggregate": compute_aggregate_metrics(rl_results),
    }
    del rl_gen, rl_mcts, rl_results
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Print results table
print("\n" + "="*80)
print("EXPERIMENT RESULTS")
print("="*80)
print(f"{'Method':<20} {'Accuracy':<12} {'Correct':<10} {'Avg Tokens':<12} {'Avg Time':<12} {'Prune Rate':<12}")
print("-"*80)

for method, data in all_results.items():
    agg = data["aggregate"]
    acc = agg.get("accuracy", 0)
    correct = agg.get("n_correct", 0)
    total = agg.get("n_total", 0)
    tokens = agg.get("avg_tokens", 0)
    t = agg.get("avg_time", 0)
    prune = agg.get("prune_rate", 0)
    print(f"{method:<20} {acc:<12.1%} {correct}/{total:<7} {tokens:<12.0f} {t:<12.2f}s {prune:<12.1%}")

print("="*80)

# Save results
with open("/content/experiments/results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

# Summary CSV
summary_rows = []
for method, data in all_results.items():
    agg = data["aggregate"]
    summary_rows.append({
        "Method": method, "Accuracy": agg.get("accuracy", 0),
        "Correct": agg.get("n_correct", 0), "Total": agg.get("n_total", 0),
        "Avg Tokens": agg.get("avg_tokens", 0), "Avg Time (s)": agg.get("avg_time", 0),
        "Prune Rate": agg.get("prune_rate", 0),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("/content/experiments/summary.csv", index=False)
print("\nResults saved to /content/experiments/")

---
## 7. Publication-Quality Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc", "axes.labelcolor": "#222222",
    "axes.titlesize": 15, "axes.labelsize": 12, "axes.titleweight": "bold",
    "text.color": "#222222", "grid.color": "#e0e0e0", "grid.linewidth": 0.8,
    "legend.facecolor": "white", "legend.edgecolor": "#cccccc", "legend.fontsize": 11,
    "figure.dpi": 200, "savefig.dpi": 300, "savefig.bbox": "tight",
})

PALETTE = ["#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974", "#64B5CD"]

print("Plot styling configured.")

In [ ]:
# --- SFT Training Loss Curve ---
fig, ax = plt.subplots(figsize=(11, 5.5))

if sft_train_losses:
    steps, losses = zip(*sft_train_losses)
    losses = np.array(losses); steps = np.array(steps)
    ax.plot(steps, losses, color=PALETTE[0], alpha=0.15, linewidth=0.8)
    window = max(1, len(losses) // 15)
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    s_steps = steps[window-1:]
    ax.plot(s_steps, smoothed, color=PALETTE[0], linewidth=2.8, label="Train Loss (smoothed)")
    ax.fill_between(s_steps, smoothed, alpha=0.08, color=PALETTE[0])
    ax.annotate(f"{losses[0]:.3f}", xy=(steps[0], losses[0]), fontsize=9, fontweight="bold",
                color=PALETTE[0], xytext=(15, 10), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color=PALETTE[0], lw=1.2))
    ax.annotate(f"{smoothed[-1]:.3f}", xy=(s_steps[-1], smoothed[-1]), fontsize=9, fontweight="bold",
                color=PALETTE[0], xytext=(-40, 10), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color=PALETTE[0], lw=1.2))

if sft_eval_losses:
    steps, losses = zip(*sft_eval_losses)
    ax.plot(steps, losses, color=PALETTE[2], linewidth=2.5, marker="D", markersize=7,
            markeredgecolor="white", markeredgewidth=1.5, label="Eval Loss", zorder=5)
    best_idx = np.argmin(losses)
    ax.scatter(steps[best_idx], losses[best_idx], color=PALETTE[1], s=120,
               edgecolors="white", linewidths=2, zorder=6)
    ax.annotate(f"Best: {losses[best_idx]:.3f}", xy=(steps[best_idx], losses[best_idx]),
                fontsize=9, fontweight="bold", color=PALETTE[1],
                xytext=(10, -20), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color=PALETTE[1], lw=1.2))

ax.set_xlabel("Training Step"); ax.set_ylabel("Loss")
ax.set_title("SFT Training Progress")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.5)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("/content/experiments/sft_loss_curve.png")
plt.show()

In [ ]:
# --- MCTS-RL Training Loss Curve ---
if rl_train_losses:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    steps, losses = zip(*rl_train_losses)
    losses = np.array(losses); steps = np.array(steps)
    ax.plot(steps, losses, color=PALETTE[3], alpha=0.15, linewidth=0.8)
    window = max(1, len(losses) // 15)
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    s_steps = steps[window-1:]
    ax.plot(s_steps, smoothed, color=PALETTE[3], linewidth=2.8, label="Train Loss (smoothed)")
    ax.fill_between(s_steps, smoothed, alpha=0.08, color=PALETTE[3])

    if rl_eval_losses:
        steps, losses = zip(*rl_eval_losses)
        ax.plot(steps, losses, color=PALETTE[2], linewidth=2.5, marker="D", markersize=7,
                markeredgecolor="white", markeredgewidth=1.5, label="Eval Loss", zorder=5)

    ax.set_xlabel("Training Step"); ax.set_ylabel("Loss")
    ax.set_title("MCTS-RL Training Progress")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.5)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig("/content/experiments/mcts_rl_loss_curve.png")
    plt.show()
else:
    print("No MCTS-RL training losses to plot.")

In [ ]:
# --- Accuracy Comparison Bar Chart ---
methods = list(all_results.keys())
n = len(methods)
accuracies = [all_results[m]["aggregate"].get("accuracy", 0) * 100 for m in methods]
tokens = [all_results[m]["aggregate"].get("avg_tokens", 0) for m in methods]
times = [all_results[m]["aggregate"].get("avg_time", 0) for m in methods]
prune_rates = [all_results[m]["aggregate"].get("prune_rate", 0) * 100 for m in methods]

has_prune = any(p > 0 for p in prune_rates)
cols = 3 if has_prune else 2
fig, axes = plt.subplots(1, cols, figsize=(6.5 * cols, 6))
x = np.arange(n)
colors = PALETTE[:n]

# Accuracy
bars = axes[0].bar(x, accuracies, width=0.6, color=colors, edgecolor="white", linewidth=1.5, zorder=3)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
                 f"{acc:.1f}%", ha="center", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Accuracy")
axes[0].set_xticks(x); axes[0].set_xticklabels(methods, rotation=25, ha="right")
axes[0].set_ylim(0, max(accuracies) * 1.25 if max(accuracies) > 0 else 100)
axes[0].grid(True, axis="y", alpha=0.4)
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)

# Compute cost
w = 0.3
axes[1].bar(x - w/2, tokens, width=w, color=colors, edgecolor="white", linewidth=1.5, zorder=3)
ax2 = axes[1].twinx()
ax2.bar(x + w/2, times, width=w, color=colors, edgecolor="white", linewidth=1.5, zorder=3, alpha=0.45)
axes[1].set_ylabel("Avg Tokens")
ax2.set_ylabel("Avg Time (s)", color="#888")
axes[1].set_title("Compute Cost")
axes[1].set_xticks(x); axes[1].set_xticklabels(methods, rotation=25, ha="right")
axes[1].grid(True, axis="y", alpha=0.4)

# Prune rate
if has_prune:
    bars3 = axes[2].bar(x, prune_rates, width=0.6, color=colors, edgecolor="white", linewidth=1.5, zorder=3)
    for bar, pr in zip(bars3, prune_rates):
        if pr > 0:
            axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                         f"{pr:.1f}%", ha="center", fontsize=12, fontweight="bold")
    axes[2].set_ylabel("Branches Pruned (%)")
    axes[2].set_title("Symbolic Verification Pruning")
    axes[2].set_xticks(x); axes[2].set_xticklabels(methods, rotation=25, ha="right")
    axes[2].grid(True, axis="y", alpha=0.4)
    axes[2].spines["top"].set_visible(False); axes[2].spines["right"].set_visible(False)

fig.suptitle("Experiment Results", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/content/experiments/comparison.png")
plt.show()

In [ ]:
# --- Per-Problem Heatmap ---
n_problems = len(all_results[methods[0]]["results"])
matrix = np.zeros((len(methods), n_problems))
for i, method in enumerate(methods):
    for j, result in enumerate(all_results[method]["results"]):
        matrix[i, j] = 1 if result.get("correct", False) else 0

fig, ax = plt.subplots(figsize=(max(10, n_problems * 0.45 + 3), max(3, len(methods) * 1.0 + 2)))
cmap = LinearSegmentedColormap.from_list("rg", ["#F4A6A0", "#A8D5BA"], N=2)
ax.imshow(matrix, cmap=cmap, aspect="auto", vmin=0, vmax=1)

ax.set_yticks(range(len(methods))); ax.set_yticklabels(methods, fontsize=11, fontweight="bold")
ax.set_xticks(range(n_problems)); ax.set_xticklabels(range(1, n_problems + 1), fontsize=9)
ax.set_xlabel("Problem Number")
ax.set_title("Per-Problem Results", fontsize=15, fontweight="bold", pad=15)

for i in range(len(methods)):
    for j in range(n_problems):
        sym = "O" if matrix[i,j] == 1 else "X"
        col = "#2d6a4f" if matrix[i,j] == 1 else "#9b2226"
        ax.text(j, i, sym, ha="center", va="center", fontsize=9, fontweight="bold", color=col)

for i, method in enumerate(methods):
    acc = all_results[method]["aggregate"].get("accuracy", 0) * 100
    col = "#2d6a4f" if acc >= 50 else "#9b2226" if acc < 30 else "#b45309"
    ax.text(n_problems + 0.5, i, f"{acc:.0f}%", va="center", fontsize=12, fontweight="bold", color=col)

legend_elements = [
    Patch(facecolor="#A8D5BA", edgecolor="#ccc", label="Correct"),
    Patch(facecolor="#F4A6A0", edgecolor="#ccc", label="Wrong"),
]
ax.legend(handles=legend_elements, loc="upper right", bbox_to_anchor=(1.0, -0.08), ncol=2)
ax.set_xlim(-0.5, n_problems + 1.5)
plt.tight_layout()
plt.savefig("/content/experiments/heatmap.png")
plt.show()

In [ ]:
# --- Efficiency Scatter: Accuracy vs Tokens ---
fig, ax = plt.subplots(figsize=(9, 6.5))
points = []
for i, method in enumerate(methods):
    acc = all_results[method]["aggregate"].get("accuracy", 0) * 100
    tok = all_results[method]["aggregate"].get("avg_tokens", 0)
    points.append((tok, acc, method, i))

points.sort(key=lambda p: p[0])
xs = [p[0] for p in points]; ys = [p[1] for p in points]
ax.plot(xs, ys, color="#ccc", linewidth=1.5, linestyle="--", zorder=1)

for tok, acc, method, i in points:
    color = PALETTE[i % len(PALETTE)]
    ax.scatter(tok, acc, color=color, s=280, zorder=5, edgecolors="white", linewidths=2.5)
    ax.annotate(method, (tok, acc), textcoords="offset points", xytext=(12, 10),
                fontsize=11, fontweight="bold", color=color,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=color, alpha=0.85))

ax.set_xlabel("Avg Tokens / Problem")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Efficiency vs Accuracy", fontsize=15, fontweight="bold", pad=15)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("/content/experiments/efficiency.png")
plt.show()

---
## 8. Download Everything

In [ ]:
# Save all training metrics
all_metrics = {
    "sft": {
        "model": MODEL_ID,
        "train_examples": len(sft_dataset["train"]),
        "eval_examples": len(sft_dataset["test"]),
        "eval_loss": round(sft_eval_loss, 4),
        "perplexity": round(math.exp(sft_eval_loss), 2),
        "trainable_params": trainable_params,
        "trainable_pct": round(trainable_params / total_params * 100, 2),
        "train_losses": sft_train_losses,
        "eval_losses": sft_eval_losses,
    },
    "mcts_rl": {
        "verified_solutions": len(verified_solutions),
        "train_losses": rl_train_losses,
        "eval_losses": rl_eval_losses,
    },
    "evaluation": {method: data["aggregate"] for method, data in all_results.items()},
}
with open("/content/experiments/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2, default=str)

print("All metrics saved.")

In [ ]:
# Zip everything and download
!cd /content && zip -r /content/aimo_results.zip \
    models/sft/lora_adapter/ \
    models/mcts_rl/lora_adapter/ \
    models/mcts_rl/verified_solutions.json \
    experiments/ \
    2>/dev/null

from google.colab import files
files.download("/content/aimo_results.zip")
print("\nDownload contains:")
print("  - models/sft/lora_adapter/        (SFT LoRA weights)")
print("  - models/mcts_rl/lora_adapter/    (MCTS-RL LoRA weights)")
print("  - models/mcts_rl/verified_solutions.json")
print("  - experiments/                    (results, plots, metrics)")